# TacticsGPT Part 2A: SFT LoRA And Testing

Run this after Part 1 has produced a tokenizer and base checkpoint in Drive.

## 0. Install Dependencies

Run once per Colab runtime. These packages are needed by the tokenizer, PyTorch model, progress bars, and data utilities.


In [ ]:
%pip install -q tokenizers tqdm numpy torch

## 1. Mount Drive And Project Folder

Run at the start of every Colab session. Keep `START_FROM_SCRATCH = False` to preserve existing Drive checkpoints and continue from old weights.


In [ ]:
from pathlib import Path
import os
import torch

from google.colab import drive

# Google Drive is always used so checkpoints survive Colab runtime resets.
drive.mount("/content/drive", force_remount=True)
PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")

# Keep this False for normal use. Because this notebook uses a new project folder,
# the first run starts clean, and later Colab restarts can resume from Drive.
START_FROM_SCRATCH = False

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

if START_FROM_SCRATCH:
    for target in [
        Path("checkpoints/pretrain"),
        Path("checkpoints/tokenizer"),
        Path("data/tactics_corpus.txt"),
    ]:
        if target.is_dir():
            import shutil
            shutil.rmtree(target)
        elif target.exists():
            target.unlink()
    print("Fresh run enabled: removed old generated tokenizer, cleaned corpus, and pretrain checkpoints.")
else:
    print("Resume mode: existing Drive corpus, tokenizer, and checkpoints will be kept.")

for folder in [
    "data/tokenizer",
    "checkpoints/pretrain",
    "checkpoints/tokenizer",
    "src",
]:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable Runtime > Change runtime type > GPU before full pretraining.")


## Required Model Source

### 3.5 GPT Model - `src/model.py`

Writes the decoder-only GPT architecture used by pretraining, SFT, RL, and generation.


In [ ]:
%%writefile src/model.py
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int = 8000
    context_length: int = 256
    n_layers: int = 4
    n_heads: int = 4
    d_model: int = 256
    ffn_hidden: int = 1024
    dropout: float = 0.1


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.context_length, config.context_length))
        self.register_buffer("causal_mask", mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq_len, channels = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(channels, dim=2)

        q = q.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        scores = scores.masked_fill(self.causal_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        y = weights @ v
        y = y.transpose(1, 2).contiguous().view(batch, seq_len, channels)
        return self.resid_dropout(self.out(y))


class FeedForward(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.ffn_hidden),
            nn.GELU(),
            nn.Linear(config.ffn_hidden, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, config: GPTConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.context_length, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        batch, seq_len = idx.shape
        if seq_len > self.config.context_length:
            raise ValueError(f"Sequence length {seq_len} exceeds context length {self.config.context_length}")

        positions = torch.arange(0, seq_len, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int = 80,
        temperature: float = 0.8,
        top_k: int | None = 50,
        greedy: bool = False,
    ) -> torch.Tensor:
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.context_length :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            if greedy:
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / max(temperature, 1e-6)
                if top_k is not None and top_k > 0:
                    values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < values[:, [-1]]] = float("-inf")
                probs = F.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)

            idx = torch.cat([idx, next_id], dim=1)
        return idx

## 7. SFT LoRA Stage: Upload Dataset

Upload a JSONL file where each row has `instruction` and `response` fields. This creates `data/sft_dataset.jsonl`.


In [ ]:
from pathlib import Path
from google.colab import files

Path("data").mkdir(exist_ok=True)

print("Upload your SFT JSONL file with fields: instruction, response")
uploaded = files.upload()

name = next(iter(uploaded))
Path("data/sft_dataset.jsonl").write_bytes(uploaded[name])

print("Saved to data/sft_dataset.jsonl")

## 7.2 Write The LoRA SFT Trainer

This script loads the Phase 1 pretrained base checkpoint, applies LoRA layers, and automatically resumes the latest SFT adapter checkpoint in the selected SFT output folder.


In [ ]:
%%writefile src/wandb_utils.py
import os


def add_wandb_args(parser, default_group="tacticsgpt"):
    parser.add_argument("--wandb_project", default="", help="Enable W&B logging by setting a project name.")
    parser.add_argument("--wandb_entity", default="", help="Optional W&B entity/team.")
    parser.add_argument("--wandb_run_name", default="", help="Optional W&B run name.")
    parser.add_argument("--wandb_group", default=default_group, help="Optional W&B group name.")
    parser.add_argument("--wandb_mode", default=os.environ.get("WANDB_MODE", "online"), choices=["online", "offline", "disabled"])
    parser.add_argument("--wandb_tags", default="", help="Comma-separated W&B tags.")


def init_wandb(args, stage, config):
    if not getattr(args, "wandb_project", ""):
        return None

    try:
        import wandb
    except ImportError as exc:
        raise RuntimeError("W&B logging requested. Install it with `pip install wandb`.") from exc

    tags = [tag.strip() for tag in getattr(args, "wandb_tags", "").split(",") if tag.strip()]
    return wandb.init(
        project=args.wandb_project,
        entity=args.wandb_entity or None,
        name=args.wandb_run_name or None,
        group=args.wandb_group or "tacticsgpt",
        job_type=stage,
        mode=args.wandb_mode,
        tags=tags,
        config=config,
    )


def wandb_log(run, data, step=None):
    if run is not None:
        run.log(data, step=step)


def wandb_finish(run):
    if run is not None:
        run.finish()


In [ ]:
%%writefile src/summarize_metrics.py
import argparse
import json
import math
from pathlib import Path


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def last_with(rows, *keys):
    for row in reversed(rows):
        if all(key in row and row[key] is not None for key in keys):
            return row
    return None


def reward_improvement_pct(current_reward, baseline_reward):
    if baseline_reward is None or not math.isfinite(baseline_reward) or abs(baseline_reward) < 1e-8:
        return None
    return 100.0 * (current_reward - baseline_reward) / abs(baseline_reward)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--pretrain_metrics", default="checkpoints/pretrain/metrics.jsonl")
    ap.add_argument("--sft_metrics", default="checkpoints/sft/metrics.jsonl")
    ap.add_argument("--rl_metrics", default="checkpoints/rl/metrics.jsonl")
    args = ap.parse_args()

    pretrain_rows = read_jsonl(args.pretrain_metrics)
    sft_rows = read_jsonl(args.sft_metrics)
    rl_rows = read_jsonl(args.rl_metrics)

    pretrain_eval = last_with(pretrain_rows, "val_loss", "val_perplexity")
    sft_eval = last_with(sft_rows, "eval_loss", "eval_perplexity")

    rl_eval_rows = [row for row in rl_rows if "eval_reward" in row and row["eval_reward"] is not None]
    rl_best = max(rl_eval_rows, key=lambda row: row["eval_reward"]) if rl_eval_rows else None
    rl_first = rl_eval_rows[0] if rl_eval_rows else None

    print("TacticsGPT metrics summary")
    print("=" * 28)

    if pretrain_eval:
        print(f"Pretrain final val loss: {pretrain_eval['val_loss']:.4f}")
        print(f"Pretrain final perplexity: {pretrain_eval['val_perplexity']:.2f}")
    else:
        print("Pretrain eval metrics: not found")

    if sft_eval:
        print(f"SFT final eval loss: {sft_eval['eval_loss']:.4f}")
        print(f"SFT final perplexity: {sft_eval['eval_perplexity']:.2f}")
    else:
        print("SFT eval metrics: not found")

    improvement = None
    if rl_best and rl_first:
        improvement = rl_best.get("reward_improvement_pct")
        if improvement is None:
            improvement = reward_improvement_pct(float(rl_best["eval_reward"]), float(rl_first["eval_reward"]))
        print(f"GRPO first eval reward: {float(rl_first['eval_reward']):.4f}")
        print(f"GRPO best eval reward: {float(rl_best['eval_reward']):.4f}")
        if improvement is not None:
            print(f"GRPO reward improvement: {improvement:.2f}%")
        else:
            print("GRPO reward improvement: unavailable because the first eval reward is zero")
    else:
        print("GRPO reward metrics: not found")

    if sft_eval and improvement is not None:
        print()
        print("Resume bullet:")
        print(
            "- Built TacticsGPT, a domain-specific football tactics LLM, with integrated W&B "
            "experiment tracking across pretraining, SFT, and GRPO; logged training loss, "
            f"perplexity, and reward curves, reaching {sft_eval['eval_loss']:.4f} eval loss, "
            f"{sft_eval['eval_perplexity']:.2f} perplexity, and {improvement:.1f}% GRPO reward improvement."
        )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train_sft_lora.py
import argparse, json, glob, os, re, math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

from model import GPT, GPTConfig
from wandb_utils import add_wandb_args, init_wandb, wandb_finish, wandb_log


def latest_step_checkpoint(folder, pattern):
    files = glob.glob(str(Path(folder) / pattern))
    if not files:
        return None
    def step(p):
        m = re.search(r"(\d+)\.pt$", os.path.basename(p))
        return int(m.group(1)) if m else -1
    return max(files, key=step)


def find_base_checkpoint():
    best = Path("checkpoints/pretrain/checkpoint_best.pt")
    if best.exists():
        return str(best)
    ckpt = latest_step_checkpoint("checkpoints/pretrain", "checkpoint_step_*.pt")
    if ckpt:
        return ckpt
    raise FileNotFoundError("No Phase 1 checkpoint found in checkpoints/pretrain")


class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        self.r = r
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=5 ** 0.5)
        nn.init.zeros_(self.lora_B)

        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.base(x) + (self.dropout(x) @ self.lora_A.T @ self.lora_B.T) * self.scale


def apply_lora(model, r=8, alpha=16, dropout=0.05):
    for block in model.blocks:
        block.attn.qkv = LoRALinear(block.attn.qkv, r, alpha, dropout)
        block.attn.out = LoRALinear(block.attn.out, r, alpha, dropout)
        block.ffn.net[0] = LoRALinear(block.ffn.net[0], r, alpha, dropout)
        block.ffn.net[2] = LoRALinear(block.ffn.net[2], r, alpha, dropout)

    for name, p in model.named_parameters():
        p.requires_grad = "lora_" in name

    return model


def lora_state_dict(model):
    return {k: v.cpu() for k, v in model.state_dict().items() if "lora_" in k}


def sft_checkpoint_payload(model, optimizer, step, epoch, base_ckpt, config, best_eval_loss):
    return {
        "lora_state_dict": lora_state_dict(model),
        "optimizer_state_dict": optimizer.state_dict(),
        "step": step,
        "epoch": epoch,
        "base_checkpoint": base_ckpt,
        "config": config,
        "best_eval_loss": best_eval_loss,
    }


class SFTDataset(Dataset):
    def __init__(self, path, tokenizer_path, context_length=256):
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.context_length = context_length
        self.pad_id = self.tokenizer.token_to_id("<pad>") or 0
        self.bos_id = self.tokenizer.token_to_id("<bos>")
        self.eos_id = self.tokenizer.token_to_id("<eos>")
        self.samples = []

        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                obj = json.loads(line)
                instruction = obj["instruction"].strip()
                response = obj["response"].strip()

                prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
                prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False).ids
                response_ids = self.tokenizer.encode(response, add_special_tokens=False).ids

                ids = [self.bos_id] + prompt_ids + response_ids + [self.eos_id]
                labels = [-100] * (1 + len(prompt_ids)) + response_ids + [self.eos_id]

                ids = ids[:context_length]
                labels = labels[:context_length]

                # Causal LM shift:
                # x sees tokens up to t, y asks for token t+1.
                x_ids = ids[:-1]
                y_ids = labels[1:]

                if any(x != -100 for x in y_ids):
                    self.samples.append((x_ids, y_ids))

        if not self.samples:
            raise ValueError("No usable SFT samples found.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

    def collate(self, batch):
        max_len = max(len(x[0]) for x in batch)
        xs, ys = [], []
        for ids, labels in batch:
            pad = max_len - len(ids)
            xs.append(ids + [self.pad_id] * pad)
            ys.append(labels + [-100] * pad)
        return torch.tensor(xs), torch.tensor(ys)


@torch.no_grad()
def evaluate(model, loader, device, max_batches):
    if loader is None:
        return None, None

    was_training = model.training
    model.eval()
    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        logits, _ = model(x)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-100)
        losses.append(loss.item())

    if was_training:
        model.train()

    avg_loss = sum(losses) / max(1, len(losses))
    return avg_loss, math.exp(min(avg_loss, 20))


def append_jsonl(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sft_data", default="data/sft_dataset.jsonl")
    ap.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    ap.add_argument("--base_checkpoint", default="")
    ap.add_argument("--out_dir", default="checkpoints/sft")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--context_length", type=int, default=256)
    ap.add_argument("--save_every", type=int, default=100)
    ap.add_argument("--eval_every", type=int, default=100)
    ap.add_argument("--eval_batches", type=int, default=50)
    ap.add_argument("--val_fraction", type=float, default=0.05)
    ap.add_argument("--log_every", type=int, default=10)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--r", type=int, default=8)
    ap.add_argument("--alpha", type=int, default=16)
    ap.add_argument("--dropout", type=float, default=0.05)
    add_wandb_args(ap)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    Path(args.out_dir).mkdir(parents=True, exist_ok=True)
    metrics_path = Path(args.out_dir) / "metrics.jsonl"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_ckpt = args.base_checkpoint or find_base_checkpoint()
    print("Base checkpoint:", base_ckpt)

    state = torch.load(base_ckpt, map_location=device)
    config = GPTConfig(**state["config"])

    model = GPT(config).to(device)
    model.load_state_dict(state["model_state_dict"])
    model = apply_lora(model, args.r, args.alpha, args.dropout).to(device)

    dataset = SFTDataset(args.sft_data, args.tokenizer, args.context_length)
    val_size = max(1, int(len(dataset) * args.val_fraction)) if args.val_fraction > 0 and len(dataset) > 20 else 0
    train_size = len(dataset) - val_size
    if val_size:
        generator = torch.Generator().manual_seed(args.seed)
        train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)
    else:
        train_dataset = dataset
        val_dataset = None

    loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=dataset.collate)
    val_loader = None
    if val_dataset is not None:
        val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=dataset.collate)

    print(f"SFT samples: {len(dataset):,}")
    print(f"SFT train samples: {train_size:,}")
    print(f"SFT validation samples: {val_size:,}")

    wandb_run = init_wandb(
        args,
        "sft",
        {
            "stage": "sft",
            "args": vars(args),
            "base_checkpoint": base_ckpt,
            "model": state["config"],
            "samples": len(dataset),
            "train_samples": train_size,
            "val_samples": val_size,
        },
    )

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr)

    step = 0
    start_epoch = 0
    best_eval_loss = float("inf")
    resume = latest_step_checkpoint(args.out_dir, "sft_step_*.pt")
    if resume:
        print("Resuming SFT:", resume)
        ckpt = torch.load(resume, map_location=device)
        model.load_state_dict(ckpt["lora_state_dict"], strict=False)
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        step = ckpt["step"]
        start_epoch = ckpt["epoch"]
        best_eval_loss = ckpt.get("best_eval_loss", best_eval_loss)

    model.train()
    for epoch in range(start_epoch, args.epochs):
        pbar = tqdm(loader, desc=f"SFT epoch {epoch+1}/{args.epochs}")
        for x, y in pbar:
            x, y = x.to(device), y.to(device)

            logits, _ = model(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=-100)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()

            step += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}", step=step)
            if args.log_every > 0 and step % args.log_every == 0:
                append_jsonl(metrics_path, {
                    "stage": "sft",
                    "step": step,
                    "epoch": epoch + 1,
                    "train_loss": float(loss.item()),
                    "lr": float(optimizer.param_groups[0]["lr"]),
                })
                wandb_log(
                    wandb_run,
                    {
                        "sft/train_loss": float(loss.item()),
                        "sft/lr": float(optimizer.param_groups[0]["lr"]),
                        "sft/epoch": epoch + 1,
                    },
                    step=step,
                )

            if val_loader is not None and args.eval_every > 0 and step % args.eval_every == 0:
                eval_loss, eval_ppl = evaluate(model, val_loader, device, args.eval_batches)
                print(f"\nSFT eval loss at step {step}: {eval_loss:.4f}")
                print(f"SFT eval perplexity at step {step}: {eval_ppl:.2f}")
                append_jsonl(metrics_path, {
                    "stage": "sft",
                    "step": step,
                    "epoch": epoch + 1,
                    "eval_loss": float(eval_loss),
                    "eval_perplexity": float(eval_ppl),
                    "best_eval_loss": float(min(best_eval_loss, eval_loss)),
                })
                wandb_log(
                    wandb_run,
                    {
                        "sft/eval_loss": float(eval_loss),
                        "sft/eval_perplexity": float(eval_ppl),
                        "sft/best_eval_loss": float(min(best_eval_loss, eval_loss)),
                    },
                    step=step,
                )
                if eval_loss < best_eval_loss:
                    best_eval_loss = eval_loss
                    best_path = Path(args.out_dir) / "sft_best.pt"
                    torch.save(
                        sft_checkpoint_payload(model, optimizer, step, epoch, base_ckpt, state["config"], best_eval_loss),
                        best_path,
                    )
                    print("Saved best SFT:", best_path)

            if step % args.save_every == 0:
                path = Path(args.out_dir) / f"sft_step_{step}.pt"
                torch.save(
                    sft_checkpoint_payload(model, optimizer, step, epoch, base_ckpt, state["config"], best_eval_loss),
                    path,
                )
                print("Saved", path)

    if val_loader is not None and (args.eval_every <= 0 or step % args.eval_every != 0):
        eval_loss, eval_ppl = evaluate(model, val_loader, device, args.eval_batches)
        print(f"\nFinal SFT eval loss: {eval_loss:.4f}")
        print(f"Final SFT perplexity: {eval_ppl:.2f}")
        append_jsonl(metrics_path, {
            "stage": "sft",
            "step": step,
            "epoch": args.epochs,
            "eval_loss": float(eval_loss),
            "eval_perplexity": float(eval_ppl),
            "best_eval_loss": float(min(best_eval_loss, eval_loss)),
            "final": True,
        })
        wandb_log(
            wandb_run,
            {
                "sft/eval_loss": float(eval_loss),
                "sft/eval_perplexity": float(eval_ppl),
                "sft/best_eval_loss": float(min(best_eval_loss, eval_loss)),
            },
            step=step,
        )
        if eval_loss < best_eval_loss:
            best_eval_loss = eval_loss
            best_path = Path(args.out_dir) / "sft_best.pt"
            torch.save(
                sft_checkpoint_payload(model, optimizer, step, args.epochs, base_ckpt, state["config"], best_eval_loss),
                best_path,
            )
            print("Saved best SFT:", best_path)

    final = Path(args.out_dir) / f"sft_step_{step}.pt"
    torch.save(
        sft_checkpoint_payload(model, optimizer, step, args.epochs, base_ckpt, state["config"], best_eval_loss),
        final,
    )
    print("Saved final SFT:", final)
    if best_eval_loss < float("inf"):
        print(f"Best SFT eval loss: {best_eval_loss:.4f}")
        print(f"Best SFT perplexity: {math.exp(min(best_eval_loss, 20)):.2f}")
    wandb_finish(wandb_run)


if __name__ == "__main__":
    main()


## 7.3 Run SFT: Smoke Test

Resume Disclaimer: this cell loads the pretrained base checkpoint first. If `checkpoints/sft/sft_step_*.pt` already exists, the trainer resumes the latest SFT adapter instead of starting SFT from zero.


In [ ]:
print("Resume Disclaimer: SFT loads the pretrained base and resumes latest checkpoints/sft/sft_step_*.pt if present.")
!python src/train_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir checkpoints/sft \
  --epochs 3 \
  --batch_size 8 \
  --lr 5e-5 \
  --context_length 256 \
  --save_every 50


## 7.4 Check Required SFT Files

Run this after mounting Drive if a later SFT cell cannot find the old corpus, tokenizer, base checkpoint, or trainer files.


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path("/content/drive/MyDrive/TacticsGPT_Phase1_Full_Pretrain")
os.chdir(PROJECT_DIR)

required = [
    "data/sft_dataset.jsonl",
    "checkpoints/tokenizer/tokenizer.json",
    "checkpoints/pretrain/checkpoint_best.pt",
    "src/train_sft_lora.py",
    "src/model.py",
]

for p in required:
    path = Path(p)
    print(p, "OK" if path.exists() else "MISSING")

## 7.5 Continue SFT: Main Run

Resume Disclaimer: this uses the same `checkpoints/sft` folder as the smoke test, so it resumes the latest SFT adapter if one exists.


In [ ]:
print("Resume Disclaimer: SFT loads the pretrained base and resumes latest checkpoints/sft/sft_step_*.pt if present.")
!python src/train_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir checkpoints/sft \
  --epochs 20 \
  --batch_size 8 \
  --lr 1e-4 \
  --context_length 256 \
  --save_every 100


## 7.6 Write The SFT Evaluation Script

Creates a small evaluation script that loads the base model plus the latest SFT LoRA adapter.


In [ ]:
%%writefile src/evaluate_sft_lora.py
import argparse, glob, os, re, json, math
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tokenizers import Tokenizer

from model import GPT, GPTConfig
from train_sft_lora import apply_lora, SFTDataset


def latest_step_checkpoint(folder, pattern):
    files = glob.glob(str(Path(folder) / pattern))
    if not files:
        return None

    def step(p):
        m = re.search(r"(\d+)\.pt$", os.path.basename(p))
        return int(m.group(1)) if m else -1

    return max(files, key=step)


@torch.no_grad()
def eval_loss(model, dataset, batch_size, device, max_batches):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=dataset.collate)
    model.eval()

    losses = []
    for i, (x, y) in enumerate(loader):
        if i >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        logits, _ = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            ignore_index=-100,
        )
        losses.append(loss.item())

    avg = sum(losses) / max(1, len(losses))
    return avg, math.exp(min(avg, 20))


@torch.no_grad()
def generate_answer(model, tokenizer, instruction, device, max_new_tokens=160, temperature=0.75, top_k=40):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False).ids
    bos_id = tokenizer.token_to_id("<bos>")

    if bos_id is not None:
        ids = [bos_id] + ids

    x = torch.tensor([ids], dtype=torch.long, device=device)

    out = model.generate(
        x,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        greedy=False,
    )

    text = tokenizer.decode(out[0].tolist(), skip_special_tokens=True)

    if "### Response:" in text:
        text = text.split("### Response:", 1)[1].strip()

    return text


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sft_data", default="data/sft_dataset.jsonl")
    ap.add_argument("--tokenizer", default="checkpoints/tokenizer/tokenizer.json")
    ap.add_argument("--sft_dir", default="checkpoints/sft")
    ap.add_argument("--checkpoint", default="")
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--max_batches", type=int, default=50)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"

    sft_ckpt = args.checkpoint or latest_step_checkpoint(args.sft_dir, "sft_step_*.pt")
    if not sft_ckpt:
        raise FileNotFoundError("No SFT checkpoint found.")

    ckpt = torch.load(sft_ckpt, map_location=device)
    base_ckpt_path = ckpt["base_checkpoint"]

    print("SFT checkpoint:", sft_ckpt)
    print("Base checkpoint:", base_ckpt_path)

    base_state = torch.load(base_ckpt_path, map_location=device)
    config = GPTConfig(**base_state["config"])

    model = GPT(config).to(device)
    model.load_state_dict(base_state["model_state_dict"])

    model = apply_lora(model).to(device)
    model.load_state_dict(ckpt["lora_state_dict"], strict=False)

    tokenizer = Tokenizer.from_file(args.tokenizer)

    dataset = SFTDataset(args.sft_data, args.tokenizer, config.context_length)
    loss, ppl = eval_loss(model, dataset, args.batch_size, device, args.max_batches)

    print(f"\nSFT eval loss: {loss:.4f}")
    print(f"SFT perplexity: {ppl:.2f}")

    prompts = [
        "How should a 4-4-2 defend centrally against a 4-2-3-1?",
        "How can a team attack a compact 5-3-2 low block?",
        "What pressing cues should a 4-3-3 use against a back three?",
        "How should full-backs behave during defensive transitions?",
        "How should a team defend crosses in a low block?",
    ]

    for p in prompts:
        print("\n" + "=" * 80)
        print("INSTRUCTION:")
        print(p)
        print("\nMODEL RESPONSE:")
        print(generate_answer(model, tokenizer, p, device))


if __name__ == "__main__":
    main()

## 7.7 Evaluate The Latest SFT Adapter

This is load-only evaluation. It does not train or overwrite SFT weights.


In [ ]:
!python src/evaluate_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --sft_dir checkpoints/sft \
  --batch_size 8 \
  --max_batches 50

## 7.8 Continue SFT: Extended Run

Resume Disclaimer: this continues from the latest `checkpoints/sft/sft_step_*.pt` adapter when that file exists.


In [ ]:
print("Resume Disclaimer: SFT continues from latest checkpoints/sft/sft_step_*.pt if present.")
!python src/train_sft_lora.py \
  --sft_data data/sft_dataset.jsonl \
  --tokenizer checkpoints/tokenizer/tokenizer.json \
  --out_dir checkpoints/sft \
  --epochs 30 \
  --batch_size 8 \
  --lr 8e-5 \
  --context_length 256 \
  --save_every 100
